In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from scipy import stats
import statsmodels.formula.api as smf


In [ ]:
# Load data
df = pd.read_excel('../data/data8.xlsx')

In [ ]:
mapping_df = pd.read_excel('../data/ValueCodings.xlsx')
mapping_df['Frage'].fillna(method='ffill',inplace=True)
mapping_df

def add_codings(df, mapping_df, col_name):
    wert_to_label = dict(zip(mapping_df[mapping_df['Frage'] == col_name]['Wert'],
                             mapping_df[mapping_df['Frage'] == col_name]['Label']))
    df[col_name + '_label'] = df[col_name].astype(str).replace(wert_to_label)
    return df

In [ ]:
mapping_df

In [ ]:
df['Q3_1c1'].value_counts()

In [ ]:
df[['Q2_1c1','new_Q2']]

In [ ]:
test = dict(zip(mapping_df[mapping_df['Frage'] == 'Q1']['Wert'],
                             mapping_df[mapping_df['Frage'] == 'Q1']['Label']))
df['test'] = df['Q1'].replace(test)
test

In [ ]:
def construct_immigrant_dummy(df):
    # Old values
    df['q2_old'] = np.where(((df['Q2_1c1'].notna())|df['Q2_1c1']=='FAIL'), np.where(df['Q2_1c1'] == 36, 0, 2), np.nan)
    df['q3_old'] = np.where(((df['Q3_1c1'].notna())|df['Q3_1c1']=='FAIL'), np.where(df['Q3_1c1'] == 36, 0, 2), np.nan)
    # generating immigrants dummy
    df['new_Q2'] = pd.to_numeric(df['new_Q2'], errors='coerce')
    df['new_Q3'] = pd.to_numeric(df['new_Q3'], errors='coerce')

    df['img'] = np.where((((df['new_Q2']== 2)&(df['new_Q3']==2)))|((df['q2_old']==2)&(df['q3_old']==2)), 1, 0)
    return df   
df = construct_immigrant_dummy(df)
df['img'].value_counts()

### Age, Gender, Where do they come from?

In [ ]:
df = add_codings(df, mapping_df, 'Q1')
df = add_codings(df, mapping_df, 'Q4_1')
df.groupby('img')[['Q1_label']].value_counts().unstack()

In [ ]:
df.groupby('img')[['Q4_1_label']].value_counts().unstack()

makes the most sense to atleast condition on the Master students. Begs the question why we din't restrict our sample on them (OR) master students in the frist place. Answer: We though twe'd have more than enough from both.

In [ ]:
df = df.loc[df['Q4_1_label']=='Master']
df.loc[df['Q4_1_label']=='Master'].groupby('img')['Q1_label'].value_counts()
(df['Q1_label'] == 'Female').groupby(df['img']).mean()

In [ ]:
df['GPA'] = df['Q13r1'].str.replace(',', '.').astype(float)
df.groupby('img')['GPA'].mean()

In [ ]:
def native_img_comparison(df, vars):
    df = df.copy()
    df_img = df[df['img']==1]
    df_nat = df[df['img']==0]

    rslt = pd.DataFrame(columns=['Immigrant Avg', 'Immigrant Std', 'Immigrant N', 'Native Avg', 'Native Std', 'Native N', 'p-value'])

    for var in vars:
        if var not in df.columns:
            raise KeyError(f"Variable '{var}' not found in DataFrame")
        df_img_var = df_img[var].dropna()
        df_nat_var = df_nat[var].dropna()
        img_mean, img_std, img_n = df_img_var.mean(), df_img_var.std(), df_img_var.count()
        nat_mean, nat_std, nat_n = df_nat_var.mean(), df_nat_var.std(), df_nat_var.count()
        _, pval = ttest_ind(df_img_var, df_nat_var, equal_var=False)
        rslt.loc[var] = [img_mean, img_std, img_n, nat_mean, nat_std, nat_n, pval]

    return rslt



variables_to_test = ['Q26r1', 'Q36r1', 'Q43r1', 'Q44r1','Q45_a', 'Q45_b', 'Q47', 'Q47_1', 'Q48r1', 'Q48_1r1','GPA']  # Add the variables you want to test

rslt = native_img_comparison(df, variables_to_test)
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

In [ ]:
rename_dict = {
    'Q26r1': 'Q26r1: What was the lowest gross annual wage you were willing to accept while searching for your first job?',
    'Q36r1':  'Q36r1: We would like to know how much you made before taxes and other deductions in the FIRST YEAR of your FIRST job, on an annual basis and in Euros. What was your gross annual salary?',
    'Q43r1':'Q43r1: In this part, we want to ask you questions about your knowledge of the corresponding job market. What do you think is the average gross annual earnings for the first full-time job for graduates from [pipe: Q6]?',
    'Q44r1': 'Q44r1: Now picture yourself in five years. We would like to know how high you expect your annual earnings (before taxes and deductions) be in five years?',
    'Q45_a': "Q45_a:  Imagine that you were forced to leave your current job and that you had 3 months to find a job at another employer in the same occupation. Do you think that you would find a job that would offer you a higher overall pay, the same pay or a lower pay",
    'Q45_b': 'Q45_b: What do you think: How much [pipe: rec_Q45] would you earn (annually and gross) in that new job?',
    'Q47': 'Q47: What percent of German graduates (on a 0-100 scale) who graduated from [pipe: Q6] do you think were working full-time immediately after graduation?',
    'Q47_1': 'Q47_1: And what percent of INTERNATIONAL (non-German) graduates do you think were working full-time immediately after graduation?',
    'Q48r1': 'Q48r1: What do you think, was the average starting gross annual salary (in euros) of natives graduates who began working full-time immediately after graduation in Germany?',
    'Q48_1r1': 'Q48_1r1: And what do you think, was the average starting gross annual salary (in euros) of international graduates who began working full-time immediately after graduation in Germany?'
}

rslt.rename(index=rename_dict, inplace=True)
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

In [ ]:
df.groupby('img')['Q36r1'].hist(legend=True,alpha=0.5)

### Removing outlier

In [ ]:
#df = df.loc[(df['Q26r1']<150) & (df['Q26r1']>=15)]
df = df.loc[(df['Q36r1']<150) & (df['Q36r1']>=15)]
rslt = native_img_comparison(df, variables_to_test)
rslt.rename(index=rename_dict, inplace=True)
rslt[['Immigrant Avg', 'Native Avg', 'p-value','Immigrant N']]

In [ ]:
df.groupby('img')['Q36r1'].hist(legend=True)

### Problems

1. Survivorship Bias: We have people who have a job and stayed in Germany. 
2. Different people altogether
3. Immigrants are a selective subset coming to Germany

### Takeaways

- Wages are not significantly different. Slightly higher for natives
- Beliefs about average pay are also not substantially different. Interesting that they think they earn less than other graduates from their program. Usually it is the opposite (Cortes et al. 2023)
- They think i) their outside options and ii) their wage in five years is substantially higher than their native counterparts.

### How many applications did you send?

1	Less than 5 applications

2	Between 5 and 10 applications

3	Between 10 and 20 applications

4	Between 20 and 50 applications

5	More than 50 applications

In [ ]:
df.groupby('img')['Q22'].value_counts(normalize=True).unstack()

Need to send considerably more applications. 

In [ ]:
df.groupby('img')['Q19r1'].mean()

Need considerably longer to find a job.

In [ ]:
df = add_codings(df, mapping_df, 'Q21')
df.groupby('img')['Q21'].value_counts(normalize=True).unstack()

### Q23: Accept first job?

In [ ]:
df.groupby('img')['Q31_1'].value_counts().unstack(0)

In [ ]:
df.groupby('img')['Q23'].value_counts(normalize=True).unstack()

More immigrants accepted the first offer

### Need to extend residence permit

In [ ]:
df.groupby('img')['Q24r1'].value_counts(normalize=True).unstack()

### Q24r3: Salary met my expectations

In [ ]:
df.groupby('img')['Q24r3'].value_counts(normalize=True).unstack()

In [ ]:
df.groupby('img')['Q45'].mean()

In [ ]:
res = smf.ols('np.log(Q36r1) ~ Q1 + img + Q19r1 + GPA + C(Q31_1)', data=df).fit(cov_type='HC1')
res.summary()

### How many employees does company of your first job have?

1	Less than 20 employees

2	Between 20 to 50 employees

3	Between 50 to 200 employees

4	Between 200 to 500 employees

5	Between 500 to 1,000 employees

6	Between 1,000 to 5,000 employees

7	Between 5,000 to 10,000 employees

8	More than 10,000

In [ ]:
df.groupby('img')['Q33'].value_counts(normalize=True).unstack()

More likely to work at very large companies but not fully coherent story here.

### Requirement of your first job?

1	No requirement

2	High School Degree

3	Vocational Training

4	Bachelor Degree

5	Master Degree or higher

6	I don’t know

In [ ]:
df.groupby('img')['Q39'].value_counts(normalize=True).unstack(0)

In [ ]:
# Neue Kodierung: 1 = Kein Universitätsabschluss erforderlich, 2 = Universitätsabschluss erforderlich, 3 = Ich weiß es nicht
def recode_q39(value):
    if value in [1, 2, 3]:
        return 0
    elif value in [4, 5]:
        return 1
    elif value == 6:
        return np.nan
    else:
        return np.nan

df['Q39_new'] = df['Q39'].apply(recode_q39)

### How many observations do we have?

In [ ]:
df.groupby(['img'])['Q39_new'].value_counts()

In [ ]:
df.groupby(['img','Q39_new'])["Q36r1"].mean().unstack(0)

Unfortunately, we only have 911 resp. 13 people in these cells because it seems that similar to our SOEP analysis we have large wage differences for jobs where no university degree is required whereas this difference is not as large when a university degree is required.

### Formal language of first job

In [ ]:
df = add_codings(df, mapping_df, 'Q32r2')
df.groupby(['img'])['Q32r2_label'].value_counts(normalize=True).unstack(0)

Immigrants have jobs where they speak English. We cannot say whether this is a preference or they weren't consider in jobs where the primary language is German. 

### Reasons for different labor market outcomes between immigrants and natives

In [ ]:
df.query('img == 1')[[f'Q52r{i}' for i in range(1,10)]].mean()

For immigrants the largest entries are 3, 4 and 9. They refer to:

- 3. Financial pressure

- 4. Language skills

- 8. Network into the industry

5 and 6 refer to application procedures. It seems they don't necessarily expect large difference there.

### What is the probability that you will be in a high/managerial position in five years?

In [ ]:
df.groupby(['img'])['Q45'].mean()

Similar to the expected wage in five years, we can document a aspiration gap where immigrants expect larger promotion jumps within the next five years. 

In [ ]:
# Neue Kodierung: 1 = Kein Universitätsabschluss erforderlich, 2 = Universitätsabschluss erforderlich, 3 = Ich weiß es nicht
def recode_gender(value):
    if value in [1,2]:
        return value
    else:
        return np.nan

df['gender'] = df['Q1'].apply(recode_gender)
df.groupby(['gender','img'])[['Q26r1','Q36r1']].mean().unstack(1)